# الفصل 49: أسس الوكلاء الذكية: ReAct والأدوات

> RAG يحل مشكلة المعرفة، لكن LLM لا يزال "عقلًا بلا يدين" — لا يستطيع إجراء حسابات دقيقة، تصفّح الإنترنت الحيّ، تنفيذ كود، أو التفاعل مع أنظمة خارجية.

- **الجزء:** 10 — التطبيقات والوكلاء الذكية
- **الفصل:** 49
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 48: تقنيات RAG المتقدمة وGraphRAG
- **الوسوم:** Agents، ReAct، Function Calling، Tools

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch49-agents-fundamentals.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-10-applications-agents%2Fch49-agents-fundamentals.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-10-applications-agents/ch49-agents-fundamentals.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch49-agents-fundamentals.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch49-agents-fundamentals.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch49-agents-fundamentals.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** requests
- **بيانات/نماذج خارجية:** بعض الخلايا تتوقع ملفات أو نماذج محمّلة مسبقًا. عدّل المسارات أو حمّل البيانات قبل التشغيل.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("requests", "requests")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch49-agents-fundamentals.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-10-applications-agents%2Fch49-agents-fundamentals.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-10-applications-agents/ch49-agents-fundamentals.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch49-agents-fundamentals.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch49-agents-fundamentals.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-10-applications-agents/ch49-agents-fundamentals.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> RAG يحل مشكلة المعرفة، لكن LLM لا يزال "عقلًا بلا يدين" — لا يستطيع إجراء حسابات دقيقة، تصفّح الإنترنت الحيّ، تنفيذ كود، أو التفاعل مع أنظمة خارجية.

> **بيانات الفصل**
>
> - **الجزء:** 10 — التطبيقات والوكلاء الذكية
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 48: تقنيات RAG المتقدمة وGraphRAG


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم نمط ReAct (Reasoning + Acting)
- فهم تعريف أدوات حقيقية وقابلة للتوسّع
- فهم Agents


## 1. لماذا يهمّك هذا الموضوع؟

RAG يحل مشكلة المعرفة، لكن LLM لا يزال "عقلًا بلا يدين" — لا يستطيع إجراء حسابات دقيقة، تصفّح الإنترنت الحيّ، تنفيذ كود، أو التفاعل مع أنظمة خارجية. الوكلاء (Agents) يمنحون LLM "أيدي" عبر **الأدوات (Tools)**، وهذا هو الاتجاه الأهم في تطبيقات LLM اليوم.


## 2. الخريطة الذهنية


```text
الوكيل البسيط = LLM + حلقة تفكير-فعل-ملاحظة (Reasoning-Acting Loop)

السؤال
   │
   ▼
[تفكير: ما الذي أحتاج فعله؟] ──Thought──┐
   │                                      │
   ▼                                      │
[فعل: استدعاء أداة معيّنة]  ──Action──────┤
   │                                      │ (تكرار حتى الحل)
   ▼                                      │
[ملاحظة: نتيجة الأداة]      ──Observation─┘
   │
   ▼
[إجابة نهائية]
```


## 3. الشرح المتدرج

### 🟢 أ) نمط ReAct (Reasoning + Acting)

**ReAct** هو النمط التأسيسي الذي بُنيت عليه معظم أطر الوكلاء الحديثة (LangChain Agents، AutoGPT، إلخ). الفكرة: نُدرِّب/نُوجِّه LLM لإنتاج تسلسل متناوب من "تفكير" (Thought) و"فعل" (Action) و"ملاحظة" (Observation).


In [ ]:
REACT_SYSTEM_PROMPT = """أنت وكيل ذكي يحل المشكلات عبر التفكير التدريجي واستخدام الأدوات المتاحة.

الأدوات المتاحة:
{tools_description}

اتّبع هذا التنسيق بدقة تامة:
السؤال: السؤال المطلوب حله
تفكير: فكّر خطوة بخطوة فيما تحتاج فعله
فعل: اسم_الأداة[المدخل المطلوب]
ملاحظة: (ستُعطى لك نتيجة الأداة تلقائيًا)
... (كرّر تفكير/فعل/ملاحظة بقدر الحاجة)
تفكير: أصبحت لدي المعلومات الكافية الآن
الإجابة النهائية: الإجابة الكاملة

ابدأ الآن:
السؤال: {question}
"""

class Tool:
    def __init__(self, name: str, description: str, function: callable):
        self.name = name
        self.description = description
        self.function = function

    def execute(self, input_str: str) -> str:
        try:
            return str(self.function(input_str))
        except Exception as e:
            return f"خطأ في تنفيذ الأداة: {str(e)}"

class ReActAgent:
    def __init__(self, llm, tools: list, max_iterations=10):
        self.llm = llm
        self.tools = {tool.name: tool for tool in tools}
        self.max_iterations = max_iterations

    def _build_tools_description(self) -> str:
        return "\n".join([f"- {t.name}: {t.description}" for t in self.tools.values()])

    def run(self, question: str) -> dict:
        tools_desc = self._build_tools_description()
        prompt = REACT_SYSTEM_PROMPT.format(tools_description=tools_desc, question=question)
        full_trace = prompt

        for iteration in range(self.max_iterations):
            response = self.llm.generate(full_trace, stop_sequences=["ملاحظة:"])
            full_trace += response

            if "الإجابة النهائية:" in response:
                final_answer = response.split("الإجابة النهائية:")[-1].strip()
                return {"answer": final_answer, "iterations": iteration + 1, "trace": full_trace}

            if "فعل:" in response:
                action_line = response.split("فعل:")[-1].strip().split("\n")[0]
                tool_name, tool_input = self._parse_action(action_line)

                if tool_name in self.tools:
                    observation = self.tools[tool_name].execute(tool_input)
                else:
                    observation = f"خطأ: الأداة '{tool_name}' غير موجودة"

                full_trace += f"\nملاحظة: {observation}\n"
            else:
                # النموذج لم يُنتج فعلًا أو إجابة نهائية — توقف أو حاول مجددًا
                break

        return {"answer": "تجاوز الوكيل الحد الأقصى للتكرارات دون الوصول لإجابة نهائية",
                "iterations": self.max_iterations, "trace": full_trace}

    def _parse_action(self, action_line: str) -> tuple:
        """تحليل سطر مثل: 'بحث_ويب[ما هي عاصمة فرنسا]'"""
        import re
        match = re.match(r'(\w+)\[(.*)\]', action_line)
        if match:
            return match.group(1), match.group(2)
        return action_line, ""


### 🟡 ب) تعريف أدوات حقيقية وقابلة للتوسّع


In [ ]:
import math
import requests

def calculator_tool(expression: str) -> str:
    """أداة حاسبة آمنة (تجنّب eval المباشر غير الآمن!)"""
    allowed_names = {k: v for k, v in math.__dict__.items() if not k.startswith("__")}
    try:
        # استخدام compile + eval بسياق محدود صارم كحل تعليمي مبسّط
        code = compile(expression, "<string>", "eval")
        for name in code.co_names:
            if name not in allowed_names:
                raise NameError(f"استخدام غير مسموح: {name}")
        return str(eval(code, {"__builtins__": {}}, allowed_names))
    except Exception as e:
        return f"خطأ في التعبير الرياضي: {str(e)}"

def web_search_tool(query: str, api_key: str = None) -> str:
    """أداة بحث ويب مبسّطة (تمثيل مفاهيمي لاستدعاء API بحث حقيقي)"""
    # في التطبيق الفعلي: استدعاء Tavily/SerpAPI/Bing Search API
    response = requests.get(
        "https://api.search-provider.com/search",
        params={"q": query, "api_key": api_key}
    )
    results = response.json().get("results", [])[:3]
    return "\n".join([f"- {r['title']}: {r['snippet']}" for r in results])

def python_code_executor_tool(code: str) -> str:
    """
    تنفيذ كود بايثون في بيئة معزولة (Sandboxed)
    ⚠️ في الإنتاج: يجب استخدام sandbox حقيقي (Docker/gVisor/E2B) وليس exec مباشر!
    """
    import io, contextlib
    output_buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(output_buffer):
            exec(code, {"__builtins__": __builtins__})
        return output_buffer.getvalue() or "تم التنفيذ بنجاح دون مخرجات"
    except Exception as e:
        return f"خطأ في التنفيذ: {str(e)}"

# تجميع الأدوات للوكيل
agent_tools = [
    Tool("حاسبة", "يحسب تعبيرات رياضية. المدخل: تعبير رياضي بصيغة بايثون", calculator_tool),
    Tool("بحث_ويب", "يبحث في الإنترنت عن معلومات حديثة. المدخل: استعلام البحث",
         lambda q: web_search_tool(q, api_key="YOUR_KEY")),
    Tool("تنفيذ_كود", "ينفّذ كود بايثون ويُعيد المخرجات. المدخل: كود بايثون",
         python_code_executor_tool),
]


> ⚠️ **خطأ أمني خطير جدًا**: استخدام `exec()` أو `eval()` مباشرة بدون عزل (sandboxing) حقيقي في الإنتاج هو **ثغرة أمنية كارثية** — أي مستخدم خبيث قادر على التلاعب بالـ prompt (عبر Prompt Injection كما رأينا في الفصل 44) يمكنه تنفيذ كود عشوائي على خادمك. استخدم **دائمًا** بيئات تنفيذ معزولة حقيقية (Docker containers، gVisor، أو خدمات متخصصة مثل E2B).

### 🔴 ج) طبقة التعمّق: Function Calling المُهيكَل (مقابل ReAct النصي الحر)

النماذج الحديثة (GPT-4، Claude، Gemini، Qwen) تدعم **Function Calling** أصيل عبر تدريب خاص، وهو أكثر موثوقية من تحليل نص ReAct الحر:


In [ ]:
import json

# تعريف الأدوات بصيغة JSON Schema (المعيار المتبع في OpenAI/Anthropic APIs)
function_schemas = [
    {
        "name": "get_weather",
        "description": "الحصول على حالة الطقس الحالية لمدينة معيّنة",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "اسم المدينة"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
            },
            "required": ["city"]
        }
    }
]

class FunctionCallingAgent:
    """
    محاكاة مفاهيمية لآلية Function Calling الحديثة
    النموذج يُنتج استدعاء دالة بصيغة JSON مُهيكَلة بدلاً من نص حر يحتاج تحليلًا (parsing)
    """
    def __init__(self, llm_client, available_functions: dict, function_schemas: list):
        self.llm_client = llm_client
        self.available_functions = available_functions
        self.function_schemas = function_schemas

    def run(self, user_message: str, max_turns=5) -> str:
        messages = [{"role": "user", "content": user_message}]

        for _ in range(max_turns):
            response = self.llm_client.chat_completion(
                messages=messages,
                functions=self.function_schemas,   # يُرسَل للـ API مباشرة
                function_call="auto"
            )

            if response.get("function_call"):
                func_name = response["function_call"]["name"]
                func_args = json.loads(response["function_call"]["arguments"])

                if func_name in self.available_functions:
                    result = self.available_functions[func_name](**func_args)
                else:
                    result = f"خطأ: دالة غير معروفة {func_name}"

                messages.append({"role": "assistant", "function_call": response["function_call"]})
                messages.append({"role": "function", "name": func_name, "content": str(result)})
            else:
                return response["content"]   # إجابة نهائية، لا حاجة لأداة أخرى

        return "تجاوز الحد الأقصى من جولات استدعاء الأدوات"


## 4. مثال محلول خطوة بخطوة

**المهمة:** «سعر قطعة 240 ريالًا، عليها خصم 15%، وأريد 3 قطع. كم أدفع؟»

**مسار ReAct المتوقع:**

| الخطوة | النوع | المحتوى |
|---|---|---|
| 1 | تفكير | أحتاج حساب السعر بعد الخصم ثم الضرب في 3 |
| 2 | فعل | `حاسبة[240 * 0.85]` |
| 3 | ملاحظة | `204.0` |
| 4 | فعل | `حاسبة[204.0 * 3]` |
| 5 | ملاحظة | `612.0` |
| 6 | إجابة نهائية | 612 ريالًا |

**المقارنة مع Function Calling:** النموذج المُهيكَل يُنتج مباشرة:


```json
{
  "name": "calculator",
  "arguments": {"expression": "240 * 0.85 * 3"}
}
```


ثم تعيد الأداة `612.0`، فيصوغ النموذج الإجابة. لاحظ الفرق: ReAct يشرح كل خطوة نصيًا، بينما Function Calling يفصل «القرار» عن «التنفيذ» في JSON قابل للتحقق. الأخير أقل عرضة لأخطاء التحليل، لكنه أقل شفافية في معرفة «لماذا» اختار النموذج الأداة.

**حساب التكلفة:** إذا كان كل استدعاء LLM يستهلك 300 token متوسطًا، فمسار ReAct أعلاه استهلك نحو 1800 token (6 خطوات)، بينما Function Calling قد يحتاج استدعاءين فقط (600 token). في المهام المتكررة، هذا فرق تكلفة كبير.


## 5. نصائح وحيل

> 💡 **اجعل وصف الأداة عقدًا لا إعلانًا**: اكتب ماذا تفعل، ومتى لا تستخدمها، وشكل المدخلات والمخرجات، وأمثلة قصيرة. النموذج يختار الأداة بناءً على هذا الوصف وحده.

> 💡 **أضف `timeout` و`max_retries` لكل أداة**: أي أداة تتصل بشبكة أو تنفّذ كودًا يجب أن تفشل بسرعة وبشكل متوقع، لا أن تُعلّق الوكيل.

> 💡 **اجعل الأدوات idempotent حيث أمكن**: إذا أعاد الوكيل محاولة إرسال بريد أو خصم مبلغ، يجب أن يكون التنفيذ المكرر آمنًا عبر مفتاح idempotency.

> 💡 **سجّل كل استدعاء**: الاسم، المدخلات، المخرجات، الزمن، والتكلفة. بدون هذا السجل لا يمكن تشخيص «لماذا أجاب الوكيل بهذا الشكل».

> 💡 **حدّد سقفًا للتكرارات وسقفًا للتكلفة**: `max_iterations=8` و`max_cost_usd` لكل طلب. الوكيل الذي لا يتوقف ليس ذكيًا، بل عطل تشغيلي.


## 6. أخطاء شائعة ومصائد

> ⚠️ **تنفيذ `eval` أو `exec` دون عزل**: أخطر خطأ في هذا الفصل. أي Prompt Injection قد يحوّل «أداة حساب» إلى تنفيذ أوامر على الخادم. استخدم sandbox حقيقيًا (Docker/gVisor/E2B) وقيّد الشبكة والملفات.

> ⚠️ **الوثوق بمخرجات الأداة كتعليمات**: نتيجة بحث ويب أو مستند قد تحتوي نصًا خبيثًا يقول «تجاهل التعليمات السابقة». تعامل مع كل مخرجات الأدوات كبيانات غير موثوقة، لا كأوامر.

> ⚠️ **أدوات بصلاحيات مفتوحة**: أداة قراءة الملفات يجب أن تُقيَّد بمجلد محدد، وأداة قاعدة البيانات يجب أن تستخدم مستخدمًا للقراءة فقط حيث يكفي.

> ⚠️ **حلقات لا نهائية**: نموذج يعيد نفس الأداة بنفس المدخلات مرارًا. أضف كشف التكرار (نفس الاستدعاء 3 مرات = توقف) ورسالة تصحيح.

> ⚠️ **إغفال فشل الأدوات**: إذا أعادت الأداة خطأً، يجب أن يراه النموذج بوضوح ويحاول مسارًا بديلًا، لا أن يخترع نتيجة. اختبر مسارات الفشل لا النجاح فقط.


## 7. جدول مقارنة

| المعيار | ReAct (نص حر) | Function Calling (مُهيكَل) |
|---|---|---|
| **الموثوقية** | متوسطة (يعتمد على تحليل نص قد يفشل) | عالية (يعتمد على تدريب خاص بصيغة JSON) |
| **سهولة التطبيق** | يعمل مع أي LLM عبر الـ Prompt فقط | يتطلب دعم API/تدريب خاص |
| **الشفافية (قابلية التتبّع)** | عالية جدًا (كل خطوة تفكير مرئية) | أقل شفافية في "سبب" اختيار الأداة |
| **المرونة** | عالية (أي بنية أداة ممكنة) | مقيّد بصيغة JSON Schema |
| **الاستخدام المُفضَّل** | نماذج مفتوحة المصدر، تصحيح الأخطاء | تطبيقات إنتاجية تحتاج موثوقية عالية |


## 8. ملخص أفضل الممارسات

- اجعل وصف كل أداة **دقيقًا وواضحًا جدًا** — النموذج يختار الأداة بناءً فقط على هذا الوصف.
- قيّد صلاحيات كل أداة لأقل حد ممكن (مبدأ Least Privilege، خاصة لأدوات تنفيذ كود أو وصول لأنظمة خارجية).
- أضف **timeout** وحدود موارد صارمة لكل أداة لمنع حلقات لا نهائية أو استهلاك موارد مفرط.
- سجّل (log) كل استدعاء أداة بالكامل لأغراض التدقيق والتصحيح.

---


## 9. أسئلة مقابلات

**سؤال 1: ما الفرق بين ReAct وFunction Calling؟**

> **إجابة نموذجية:** ReAct نمط prompting يطلب من النموذج إنتاج تسلسل نصي من Thought/Action/Observation، ويحتاج محللًا نصيًا لاستخراج الأداة والمدخلات. Function Calling قدرة مُدرَّبة في النموذج لإنتاج استدعاء دالة بصيغة JSON وفق schema، فتكون أكثر موثوقية وأسهل تكاملًا. ReAct أوضح في التتبع وأكثر مرونة مع النماذج المفتوحة، بينما Function Calling أفضل للإنتاج حين يتوفر الدعم. كثير من الأنظمة الحديثة تدمجهما: تفكير حر + استدعاءات مُهيكَلة.

**سؤال 2: كيف تمنع الوكيل من الدوران في حلقة لا نهائية؟**

> **إجابة نموذجية:** عدة طبقات: حد أقصى صارم للتكرارات، كشف الاستدعاءات المتكررة بنفس المدخلات، ميزانية tokens/تكلفة لكل طلب، ومهلة زمنية إجمالية. إضافة إلى ذلك، أُدرّب الـ prompt على أن يقول «لا أملك معلومات كافية» بدلًا من تكرار المحاولة، وأُعيد توجيه النموذج برسالة واضحة عند الفشل. المراقبة تكشف الحلقات قبل أن تصل إلى المستخدم.

**سؤال 3: كيف تؤمّن تنفيذ الأدوات في الإنتاج؟**

> **إجابة نموذجية:** Least Privilege لكل أداة، sandbox للكود، قائمة بيضاء للأوامر والمواقع، عزل شبكة، حدود موارد (CPU/RAM/زمن)، تحقق من صحة المدخلات وفق schema، تسجيل وتدقيق كل استدعاء، ومراجعة بشرية للأدوات ذات الأثر المالي أو القانوني. لا تعتمد على «النموذج لن يفعل شيئًا سيئًا»؛ افترض أن المدخلات ستُهاجَم.


## 10. تمرين عملي

**الهدف:** بناء وكيل أدوات صغير وتقييمه على 20 مهمة عربية.

**الخطوات:**
1. عرّف ثلاث أدوات: حاسبة آمنة، بحث في قاعدة معرفة محلية، وتحويل تاريخ.
2. اكتب 20 مهمة تشمل: حسابًا بسيطًا، سؤالًا معرفيًا، مهمة تحتاج أداتين، ومهمة مستحيلة.
3. نفّذ الوكيل بنمطين: ReAct نصي وFunction Calling إن توفر دعم API.
4. سجّل لكل مهمة: النجاح/الفشل، عدد الاستدعاءات، الزمن، والتكلفة.
5. احسب: معدل النجاح، متوسط الاستدعاءات، ونسبة المهام التي توقفت بسبب الحد الأقصى.
6. أضف هجوم Prompt Injection في نتيجة أداة البحث، وتأكد أن الوكيل لا ينفّذه.

**معيار النجاح:** نجاح 16/20 على الأقل، وصفر تنفيذ لأوامر خبيثة، ومتوسط أقل من 5 استدعاءات لكل مهمة.


## 11. مراجع للتعمّق

- Yao, S., et al. (2022). *ReAct: Synergizing Reasoning and Acting in Language Models*. ICLR 2023.
- Schick, T., et al. (2023). *Toolformer: Language Models Can Teach Themselves to Use Tools*. NeurIPS.
- OpenAI. *Function Calling Guide*. platform.openai.com/docs.
- Anthropic. *Tool Use (Function Calling) Documentation*. docs.anthropic.com.
- Qin, Y., et al. (2023). *ToolLLM: Facilitating Large Language Models to Master 16000+ Real-world APIs*. ICLR 2024.


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التطبيقات والوكلاء الذكية / الفصل 49: أسس الوكلاء الذكية: ReAct والأدوات.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
